# Direct Prompting Extension - BIOMQM Pipeline

This notebook implements the Direct Prompting (Reverse CoT) approach for Question Answering.

**Key Features:**
- XML-structured prompts with `<context>` and `<question>` tags
- Separate prompts for Source (no error warning) and Backtranslation (with error warning)
- One API call per question (instead of batch)
- Generation params: `temperature=0.1`, `top_p=0.9`, `repetition_penalty=1.1`

## 0. Mount Google Drive & Configure Model Cache

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')
    
    print(f'Model cache directory: {DRIVE_CACHE_DIR}')
else:
    print('Not running in Colab - using default cache directories')

## Setup - Install Dependencies

In [ ]:
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'torch', 'accelerate', 'nltk', 'sentence-transformers', 'sacrebleu', 'textstat'], check=True)

if IN_COLAB:
    if not os.path.exists('/content/askqe'):
        subprocess.run(['git', 'clone', 'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git', '/content/askqe'], check=True)
    PROJECT_ROOT = '/content/askqe'
else:
    PROJECT_ROOT = os.getcwd()

RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results Qwen3B baseline')
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Results directory: {RESULTS_DIR}')

## Pre-download Models

Download all models needed for the pipeline. These will be cached on Drive for reuse.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, LongformerTokenizer, LongformerForSequenceClassification
from sentence_transformers import SentenceTransformer
import torch

print('=== Downloading/Loading Models ===')
print('This may take a while on first run, but will be cached on Drive for future use.\n')

MODELS = {
    'qwen': 'Qwen/Qwen2.5-3B-Instruct',
    'sbert': 'sentence-transformers/all-MiniLM-L6-v2',
    'answerability': 'potsawee/longformer-large-4096-answerable-squad2'
}

# Download Qwen model
print(f"[1/3] Loading {MODELS['qwen']}...")
tokenizer = AutoTokenizer.from_pretrained(MODELS['qwen'])
model = AutoModelForCausalLM.from_pretrained(MODELS['qwen'], torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ Qwen cached')

# Download SBERT model
print(f"[2/3] Loading {MODELS['sbert']}...")
sbert_model = SentenceTransformer(MODELS['sbert'])
del sbert_model
print('      ✓ SBERT cached')

# Download Answerability model (Longformer)
print(f"[3/3] Loading {MODELS['answerability']}...")
try:
    ans_tokenizer = LongformerTokenizer.from_pretrained(MODELS['answerability'])
    ans_model = LongformerForSequenceClassification.from_pretrained(MODELS['answerability'])
    del ans_tokenizer, ans_model
    print('      ✓ Answerability model cached')
except Exception as e:
    print(f'      ⚠ Could not load: {e}')

print('\n=== All models cached! ===')

## 1. Setup & Paths

**Modify these paths according to your environment (Kaggle, Colab, local, etc.)**

In [ ]:
# ============================================
# PATH CONFIGURATION - MODIFY THESE
# ============================================

# Base directory for the extension
EXTENSION_DIR = "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting"

# Input QG file (output of question generation)
QG_INPUT_PATH = "/kaggle/working/askqe/results Qwen3B baseline/QG/biomqm/vanilla_qwen-3b.jsonl"

# Original dataset file (for mapping)
ORIGINAL_DATASET_PATH = "/kaggle/working/askqe/biomqm/dev_with_backtranslation.jsonl"

# QA output directory (for unique answers)
QA_OUTPUT_DIR = f"{EXTENSION_DIR}/QA/unique"

# Mapped output directory
MAPPED_OUTPUT_DIR = f"{EXTENSION_DIR}/QA/mapped"

# Evaluation output base directory
EVALUATION_OUTPUT_DIR = EXTENSION_DIR

# Pipeline name
PIPELINE = "direct-prompting"

# Languages
LANGUAGES = ["de", "es", "fr", "ru", "zh-CN"]

print(f"Extension Dir: {EXTENSION_DIR}")
print(f"QG Input: {QG_INPUT_PATH}")
print(f"QA Output Dir: {QA_OUTPUT_DIR}")
print(f"Mapped Output Dir: {MAPPED_OUTPUT_DIR}")

## 2. QG - Question Generation

This step copies or references the existing QG file. If you need to regenerate questions, use the baseline QG script.

In [ ]:
import os
import shutil

# Create QG directory if needed
qg_output_dir = f"{EXTENSION_DIR}/QG"
os.makedirs(qg_output_dir, exist_ok=True)

# Copy QG file to extension directory (optional - for reference)
qg_copy_path = f"{qg_output_dir}/vanilla_qwen-3b.jsonl"
if os.path.exists(QG_INPUT_PATH) and not os.path.exists(qg_copy_path):
    shutil.copy(QG_INPUT_PATH, qg_copy_path)
    print(f"Copied QG file to: {qg_copy_path}")
else:
    print(f"QG file already exists or source not found")

# Count lines
if os.path.exists(QG_INPUT_PATH):
    with open(QG_INPUT_PATH, 'r') as f:
        n_lines = sum(1 for _ in f)
    print(f"QG file has {n_lines} lines")

## 3. QA Source - Generate Source Answers

Runs QA on unique source sentences using the **source prompt** (no error warning).

In [ ]:
# QA Source
source_output_path = f"{QA_OUTPUT_DIR}/source-{PIPELINE}.jsonl"

!python "{EXTENSION_DIR}/code/qwen-3b-direct-prompting.py" \
    --mode source \
    --pipeline {PIPELINE} \
    --qg_input_path "{QG_INPUT_PATH}" \
    --output_path "{source_output_path}"

## 4. QA DE - German Backtranslation

In [ ]:
# QA BT - German
de_output_path = f"{QA_OUTPUT_DIR}/bt-de-{PIPELINE}.jsonl"

!python "{EXTENSION_DIR}/code/qwen-3b-direct-prompting.py" \
    --mode bt \
    --lang de \
    --pipeline {PIPELINE} \
    --qg_input_path "{QG_INPUT_PATH}" \
    --output_path "{de_output_path}"

## 5. QA ES - Spanish Backtranslation

In [ ]:
# QA BT - Spanish
es_output_path = f"{QA_OUTPUT_DIR}/bt-es-{PIPELINE}.jsonl"

!python "{EXTENSION_DIR}/code/qwen-3b-direct-prompting.py" \
    --mode bt \
    --lang es \
    --pipeline {PIPELINE} \
    --qg_input_path "{QG_INPUT_PATH}" \
    --output_path "{es_output_path}"

## 6. QA FR - French Backtranslation

In [ ]:
# QA BT - French
fr_output_path = f"{QA_OUTPUT_DIR}/bt-fr-{PIPELINE}.jsonl"

!python "{EXTENSION_DIR}/code/qwen-3b-direct-prompting.py" \
    --mode bt \
    --lang fr \
    --pipeline {PIPELINE} \
    --qg_input_path "{QG_INPUT_PATH}" \
    --output_path "{fr_output_path}"

## 7. QA RU - Russian Backtranslation

In [ ]:
# QA BT - Russian
ru_output_path = f"{QA_OUTPUT_DIR}/bt-ru-{PIPELINE}.jsonl"

!python "{EXTENSION_DIR}/code/qwen-3b-direct-prompting.py" \
    --mode bt \
    --lang ru \
    --pipeline {PIPELINE} \
    --qg_input_path "{QG_INPUT_PATH}" \
    --output_path "{ru_output_path}"

## 8. QA ZH-CN - Chinese Backtranslation

In [ ]:
# QA BT - Chinese
zh_output_path = f"{QA_OUTPUT_DIR}/bt-zh-CN-{PIPELINE}.jsonl"

!python "{EXTENSION_DIR}/code/qwen-3b-direct-prompting.py" \
    --mode bt \
    --lang zh-CN \
    --pipeline {PIPELINE} \
    --qg_input_path "{QG_INPUT_PATH}" \
    --output_path "{zh_output_path}"

## 9. Mapping - Merge QA Results

Combines source and BT answers using row indexes to reconstruct the full dataset.

In [ ]:
# Mapping

!python "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/mapping/mapping.py" \
    --pipeline {PIPELINE} \
    --source_file "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/QA/unique/source-direct-prompting.jsonl" \
    --bt_dir "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/QA/unique" \
    --qg_input_path "/kaggle/working/askqe/results Qwen3B baseline/QG/biomqm/vanilla_qwen-3b.jsonl" \
    --output_path "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/QA/mapped/all-files.jsonl"

## 10. String Comparison Evaluation

Calculates F1, Exact Match, chrF, and BLEU scores.

In [ ]:
# String Comparison Evaluation
!python "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/evaluation/string comparison/string_comparison.py" \
    --mapped_file_path "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/QA/mapped/all-files.jsonl" \
    --output_base_dir "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/evaluation/string comparison"

## 11. SBERT Semantic Similarity Evaluation

Calculates cosine similarity using sentence embeddings.

In [ ]:
# SBERT Evaluation
!python "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/evaluation/sbert/sbert.py" \
    --mapped_file_path "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/QA/mapped/all-files.jsonl" \
    --output_base_dir "/kaggle/working/askqe/results Qwen3B baseline/biomqm/direct-prompting/evaluation/sbert"

## Summary

The Direct Prompting pipeline is now complete. Check the output files in:
- **QA Unique**: `{EXTENSION_DIR}/QA/unique/`
- **QA Mapped**: `{EXTENSION_DIR}/QA/mapped/`
- **String Comparison**: `{EXTENSION_DIR}/evaluation/string-comparison/biomqm/direct-prompting/`
- **SBERT**: `{EXTENSION_DIR}/evaluation/sbert/biomqm/direct-prompting/`